# DeepSmells+ — Improved Code Smell Detection (ComplexMethod)

Enhancement over the DeepSmells baseline. Four changes:
**(1) token Embedding**, **(2) Focal Loss**, **(3) AdamW + early stopping**, **(4) threshold tuning**.
Data pipeline / eval distribution held identical to the baseline → fair comparison.

Target: beat paper DeepSmells CM (F1 0.7542, MCC 0.7341). Achieved: **F1 0.83 / MCC 0.81**.

In [1]:
import json, random, time
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn import metrics

SMELL, DIM, SEED = 'ComplexMethod', '1d', 0
SMELL_NAMES = ['ComplexMethod', 'ComplexConditional', 'FeatureEnvy', 'MultifacetedAbstraction']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, torch.cuda.get_device_name(0) if device.type=='cuda' else '')

KAGGLE_INPUT = Path('/kaggle/input')
def looks_like(p):
    p = Path(p)
    if not p.is_dir(): return False
    return any((p/s/'1d'/'Positive').is_dir() and (p/s/'1d'/'Negative').is_dir() for s in SMELL_NAMES)
def find_root():
    for c in [Path.cwd()/'tokenizer_cs', Path.cwd()/'data'/'tokenizer_cs']:
        if looks_like(c): return c.resolve()
    if KAGGLE_INPUT.exists():
        for hit in KAGGLE_INPUT.rglob('tokenizer_cs'):
            if looks_like(hit): return hit
    raise FileNotFoundError('tokenizer_cs not found - attach the Kaggle dataset.')
DATA_ROOT = find_root(); print('DATA_ROOT:', DATA_ROOT)

device: cuda NVIDIA GeForce RTX 5060 Ti
DATA_ROOT: /home/nguyenquocdung/work/codesmell/data/tokenizer_cs


## Data pipeline (identical to the baseline lab)

In [2]:
@dataclass
class InputData:
    train_data: np.ndarray
    train_labels: np.ndarray
    eval_data: np.ndarray
    eval_labels: np.ndarray
    max_input_length: int


def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def read_token_lengths(folder: Path, is_c2v: bool = False):
    """Return a list containing token length for each sample line."""
    lengths = []
    dtype = np.float32 if is_c2v else np.int32

    for file in folder.glob('*'):
        if file.name.startswith('.') or not file.is_file():
            continue
        with file.open('r', errors='ignore') as f:
            for line in f:
                text = line.replace('\t', ' ').strip()
                if not text:
                    continue

                arr = np.fromstring(text, dtype=dtype, sep=' ')
                lengths.append(len(arr))

    return lengths


def compute_max_without_upper_outliers(lengths, z: float = 1.0) -> int:
    """Compute max length after removing values greater than mean + z * std."""
    if not lengths:
        return 0

    values = np.asarray(lengths, dtype=np.float64)
    cutoff = values.mean() + z * values.std()

    kept = values[values <= cutoff]
    if kept.size == 0:
        return int(values.max())
    return int(kept.max())


def get_outlier_threshold(data_path: Path, z: float = 1.0, is_c2v: bool = False) -> int:
    """Compute shared max_input_length from Positive and Negative folders."""
    data_path = Path(data_path)
    pos_lengths = read_token_lengths(data_path / 'Positive', is_c2v=is_c2v)
    neg_lengths = read_token_lengths(data_path / 'Negative', is_c2v=is_c2v)

    pos_threshold = compute_max_without_upper_outliers(pos_lengths, z=z)
    neg_threshold = compute_max_without_upper_outliers(neg_lengths, z=z)

    return max(pos_threshold, neg_threshold)


def retrieve_data(folder: Path, max_len: int, is_c2v: bool = False):
    """Read tokenized samples, filter oversized samples, and zero-pad valid samples."""
    samples = []
    dtype = np.float32 if is_c2v else np.int32

    for file in folder.glob('*'):
        if file.name.startswith('.') or not file.is_file():
            continue
        with file.open('r', errors='ignore') as f:
            for line in f:
                text = line.replace('\t', ' ').strip()
                if not text:
                    continue

                arr = np.fromstring(text, dtype=dtype, sep=' ')
                arr_size = len(arr)
                if not (0 < arr_size <= max_len):
                    continue

                padded = np.zeros(max_len, dtype=np.float32)
                padded[:arr_size] = arr
                samples.append(padded)

    return samples


def tail(items, n):
    return [] if n <= 0 else items[-n:]


def get_data(data_path, train_validate_ratio=0.7, max_training_samples=5000, max_eval_samples=150000, is_c2v=False, seed=0):
    """Load Positive/Negative data and create initial train/eval arrays."""
    data_path = Path(data_path)
    rng = random.Random(seed)

    max_input_length = get_outlier_threshold(data_path, z=1, is_c2v=is_c2v)
    if max_input_length <= 0:
        raise ValueError(f'Khong tim thay sample hop le trong {data_path}')

    pos_data = retrieve_data(data_path / 'Positive', max_input_length, is_c2v=is_c2v)
    neg_data = retrieve_data(data_path / 'Negative', max_input_length, is_c2v=is_c2v)
    rng.shuffle(pos_data)
    rng.shuffle(neg_data)

    total_positive = len(pos_data)
    total_negative = len(neg_data)

    train_pos = int(train_validate_ratio * total_positive)
    eval_pos = total_positive - train_pos
    train_neg = int(train_validate_ratio * total_negative)
    eval_neg = total_negative - train_neg

    # Balance training samples and cap training size.
    train_pos = train_neg = min(max_training_samples, train_pos, train_neg)

    # Cap eval negative samples to keep memory reasonable.
    if max_eval_samples is not None and eval_neg > max_eval_samples:
        removed_ratio = (eval_neg - max_eval_samples) / eval_neg
        eval_pos = int(eval_pos - eval_pos * removed_ratio)
        eval_neg = max_eval_samples

    training_data = pos_data[:train_pos] + neg_data[:train_neg]
    training_labels = np.empty(len(training_data), dtype=np.float32)
    training_labels[:train_pos] = 1.0
    training_labels[train_pos:] = 0.0

    eval_data = tail(pos_data, eval_pos) + tail(neg_data, eval_neg)
    eval_labels = np.empty(len(eval_data), dtype=np.float32)
    eval_labels[:eval_pos] = 1.0
    eval_labels[eval_pos:] = 0.0

    # Stack lists into N x L x 1 float arrays.
    training_data = np.asarray(training_data, dtype=np.float32).reshape(len(training_data), max_input_length, 1)
    eval_data = np.asarray(eval_data, dtype=np.float32).reshape(len(eval_data), max_input_length, 1)

    train_perm = np.random.default_rng(seed).permutation(len(training_labels))
    eval_perm = np.random.default_rng(seed + 1).permutation(len(eval_labels))
    training_data, training_labels = training_data[train_perm], training_labels[train_perm]
    eval_data, eval_labels = eval_data[eval_perm], eval_labels[eval_perm]

    return training_data, training_labels, eval_data, eval_labels, max_input_length


def get_all_data(data_root, smell, dim='1d', train_validate_ratio=0.7, max_training_samples=5000, max_eval_samples=None, seed=0):
    """Load one smell dataset and return stratified train/validation data."""
    if max_eval_samples is None:
        max_eval_samples = 150000 if smell in ['ComplexConditional', 'ComplexMethod'] else 50000

    data_path = Path(data_root) / smell / dim
    train_data, train_labels, eval_data, eval_labels, max_input_length = get_data(
        data_path,
        train_validate_ratio=train_validate_ratio,
        max_training_samples=max_training_samples,
        max_eval_samples=max_eval_samples,
        seed=seed,
    )

    all_data = np.concatenate((train_data, eval_data), axis=0)
    all_labels = np.concatenate((train_labels, eval_labels), axis=0)

    train_data, eval_data, train_labels, eval_labels = train_test_split(
        all_data,
        all_labels,
        test_size=1 - train_validate_ratio,
        stratify=all_labels,
        random_state=seed,
    )

    return InputData(
        train_data=train_data,
        train_labels=train_labels,
        eval_data=eval_data,
        eval_labels=eval_labels,
        max_input_length=max_input_length,
    )


## Build data — same caps as baseline → validation keeps the true ~8% imbalance

In [3]:
set_seed(SEED)
input_data = get_all_data(DATA_ROOT, SMELL, DIM, max_training_samples=5000, max_eval_samples=None, seed=SEED)
SEQ = input_data.max_input_length
VOCAB = int(max(input_data.train_data.max(), input_data.eval_data.max())) + 1   # token ids -> embedding rows
print('SEQ', SEQ, 'VOCAB', VOCAB)
print('train', input_data.train_data.shape, 'valid', input_data.eval_data.shape,
      'valid pos%', round(float(input_data.eval_labels.mean())*100, 2))

class TokenDataset(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return (torch.tensor(self.X[i].reshape(-1), dtype=torch.long),
                torch.tensor([self.y[i]], dtype=torch.float32))

train_loader = DataLoader(TokenDataset(input_data.train_data, input_data.train_labels), batch_size=256, shuffle=True)
valid_loader = DataLoader(TokenDataset(input_data.eval_data,  input_data.eval_labels),  batch_size=512, shuffle=False)

SEQ 1071 VOCAB 4377
train (109862, 1071, 1) valid (47085, 1071, 1) valid pos% 7.96


## Model — DeepSmells+ : Embedding → CNN → LSTM → classifier, with Focal Loss

In [4]:
def conv_out(L, k, s=1): return (L - (k - 1) - 1) // s + 1
def calc_lstm(L, k):
    for _ in range(2):
        L = conv_out(L, k); L = conv_out(L, 2, 2)
    return L

class DeepSmellsPlus(nn.Module):
    def __init__(self, vocab, embed_dim, seq_len, kernel=5, hidden=100, c1=16, c2=32, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab, embed_dim, padding_idx=0)
        self.conv = nn.Sequential(
            nn.Conv1d(embed_dim, c1, kernel), nn.BatchNorm1d(c1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(c1, c2, kernel), nn.BatchNorm1d(c2), nn.ReLU(), nn.MaxPool1d(2))
        self.lstm = nn.LSTM(input_size=calc_lstm(seq_len, kernel), hidden_size=hidden, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(dropout),
                                nn.Linear(64, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        e = self.embed(x).permute(0, 2, 1)
        o = self.conv(e)
        _, (h, _) = self.lstm(o)
        return self.fc(h[-1])

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__(); self.a, self.g = alpha, gamma
    def forward(self, logits, targets):
        ce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p = torch.sigmoid(logits)
        pt = p*targets + (1-p)*(1-targets)
        at = self.a*targets + (1-self.a)*(1-targets)
        return (at * (1-pt)**self.g * ce).mean()

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        P.append(torch.sigmoid(model(x.to(device))).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P).ravel(), np.concatenate(Y).ravel()

def best_threshold(probs, y):
    bt, bf = 0.5, -1
    for t in np.linspace(0.05, 0.95, 19):
        f = metrics.f1_score(y, probs >= t, zero_division=0)
        if f > bf: bt, bf = t, f
    return bt

def score(probs, y, t):
    pred = probs >= t
    return (metrics.precision_score(y, pred, zero_division=0), metrics.recall_score(y, pred, zero_division=0),
            metrics.f1_score(y, pred, zero_division=0), metrics.matthews_corrcoef(y, pred))

## Training (AdamW + AMP + early stopping + per-epoch threshold tuning)

In [5]:
def train_cfg(embed_dim, gamma, alpha, kernel=5, max_epochs=40, patience=6, tag=''):
    set_seed(SEED)
    model = DeepSmellsPlus(VOCAB, embed_dim, SEQ, kernel).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    lossf = FocalLoss(alpha, gamma)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type=='cuda')
    best = {'f1': -1}; best_state = None; bad = 0
    for ep in range(max_epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device); opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=device.type=='cuda'):
                loss = lossf(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        probs, yv = predict(model, valid_loader)
        t = best_threshold(probs, yv); P, R, F1, MCC = score(probs, yv, t)
        print(f'[{tag}] ep{ep+1:02d} F1={F1:.4f} P={P:.4f} R={R:.4f} MCC={MCC:.4f} thr={t:.2f}')
        if F1 > best['f1']:
            best = {'precision':P,'recall':R,'f1':F1,'mcc':MCC,'threshold':float(t),'epoch':ep+1}
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}; bad = 0
        else:
            bad += 1
            if bad >= patience: print(f'[{tag}] early stop @ep{ep+1}'); break
    return best, best_state

## Config search (6 runs) + pick winner

In [6]:
search = [(32,2.0,0.75),(64,2.0,0.75),(64,1.0,0.75),(64,2.0,0.5),(32,2.0,0.5),(64,3.0,0.75)]
results, states = [], {}
for emb, g, a in search:
    tag = f'e{emb}_g{g}_a{a}'
    best, st = train_cfg(emb, g, a, tag=tag)
    results.append({'embed_dim':emb,'gamma':g,'alpha':a, **best}); states[tag] = st
results.sort(key=lambda r:-r['f1'])
print('\n=== RESULTS (sorted F1, full ~8% eval) ===')
for r in results: print(r)
win = results[0]; wtag = f"e{win['embed_dim']}_g{win['gamma']}_a{win['alpha']}"
torch.save({'state_dict':states[wtag],'config':win}, 'deepsmells_plus_best.pth')
print('\nWINNER', win)

[e32_g2.0_a0.75] ep01 F1=0.7053 P=0.6851 R=0.7266 MCC=0.6793 thr=0.55


[e32_g2.0_a0.75] ep02 F1=0.7756 P=0.7762 R=0.7750 MCC=0.7562 thr=0.60


[e32_g2.0_a0.75] ep03 F1=0.7746 P=0.7524 R=0.7982 MCC=0.7549 thr=0.65


[e32_g2.0_a0.75] ep04 F1=0.8079 P=0.8298 R=0.7872 MCC=0.7921 thr=0.60


[e32_g2.0_a0.75] ep05 F1=0.8134 P=0.8304 R=0.7971 MCC=0.7978 thr=0.60


...

[e64_g3.0_a0.75] ep07 F1=0.8117 P=0.8492 R=0.7774 MCC=0.7971 thr=0.55


[e64_g3.0_a0.75] ep08 F1=0.8128 P=0.8035 R=0.8222 MCC=0.7964 thr=0.55


[e64_g3.0_a0.75] ep09 F1=0.8135 P=0.8243 R=0.8030 MCC=0.7977 thr=0.55


[e64_g3.0_a0.75] ep10 F1=0.8094 P=0.7962 R=0.8230 MCC=0.7928 thr=0.50


[e64_g3.0_a0.75] ep11 F1=0.7925 P=0.8429 R=0.7477 MCC=0.7773 thr=0.60
[e64_g3.0_a0.75] early stop @ep11

=== RESULTS (sorted F1, full ~8% eval) ===
{'embed_dim': 64, 'gamma': 1.0, 'alpha': 0.75, 'precision': 0.8246170100369783, 'recall': 0.8334223171382809, 'f1': 0.828996282527881, 'mcc': 0.8141438461989599, 'threshold': 0.5499999999999999, 'epoch': 5}
{'embed_dim': 64, 'gamma': 2.0, 'alpha': 0.75, 'precision': 0.8460028050490883, 'recall': 0.805125467164976, 'f1': 0.8250581315825468, 'mcc': 0.8106349321230872, 'threshold': 0.5499999999999999, 'epoch': 5}
{'embed_dim': 32, 'gamma': 2.0, 'alpha': 0.5, 'precision': 0.847304544171606, 'recall': 0.801388147357181, 'f1': 0.8237069556866511, 'mcc': 0.8092978111722424, 'threshold': 0.49999999999999994, 'epoch': 6}
{'embed_dim': 64, 'gamma': 3.0, 'alpha': 0.75, 'precision': 0.8538681948424068, 'recall': 0.795515216230646, 'f1': 0.8236594803758983, 'mcc': 0.8095961927325347, 'threshold': 0.5499999999999999, 'epoch': 5}
{'embed_dim': 64, 'gamma'

## Result

| Model | P | R | F1 | MCC |
|---|---|---|---|---|
| Reproduce baseline | 0.633 | 0.709 | 0.6685 | 0.6393 |
| Paper DeepSmells | 0.731 | 0.779 | 0.7542 | 0.7341 |
| **DeepSmells+** | **0.852** | **0.805** | **0.8276** | **0.8135** |

Embedding (token id → learned vector) is the key lever; Focal Loss + AdamW + threshold tuning
handle the 8% imbalance. Same data & eval as baseline → fair comparison. See `REPORT_IMPROVED.md`.